In [ ]:
!pip install gensim
import gensim.downloader as api

# This will download the ~1.6GB model directly into your Colab session
model = api.load("word2vec-google-news-300")

# Quick test: Find words similar to 'software'
print(model.most_similar("software", topn=5))

In [ ]:
result = model.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)
print(f"Analogy Result: {result}")

In [ ]:
print(model.doesnt_match(["apple", "microsoft", "google", "chair"]))

In [ ]:
# 1) Query expansion (useful for search engines)
# Add related terms to improve recall of user queries.
query = "car"
most_similar = model.most_similar(query, topn=8)
expanded_terms = [w for w, score in most_similar]
print(f"Original query: {query}")
print("Expanded terms:", expanded_terms)

# Example: these terms can be appended to a search query.

In [ ]:
# 2) Semantic matching for FAQ / support routing
# Choose the FAQ topic whose keyword is closest to the user issue.

# Candidate FAQ categories we want to compare against the user's issue.
faq_topics = ["billing", "refund", "password", "shipping", "account"]

# The user issue is represented by a single keyword for this simple example.
user_issue = "payment"

# Store each FAQ topic together with its Word2Vec similarity score.
similarities = []
for topic in faq_topics:
    # Only compare words that exist in the loaded Word2Vec vocabulary.
    if user_issue in model.key_to_index and topic in model.key_to_index:
        # Higher similarity means the words are closer in semantic meaning.
        sim = model.similarity(user_issue, topic)
        similarities.append((topic, sim))

# Sort topics from most similar to least similar.
similarities = sorted(similarities, key=lambda x: x[1], reverse=True)

# Print the ranked FAQ topics so the best routing option appears first.
print(f"User issue term: {user_issue}")
print("Best matching FAQ topics:")
for topic, score in similarities:
    print(f"  {topic:10s} -> {score:.3f}")

In [ ]:
# 3) Category assignment by prototype words
# A lightweight intent/topic classifier without training.

# Each category is represented by a few example words, called prototype or seed words.
categories = {
    "technology": ["computer", "software", "internet"],
    "sports": ["football", "basketball", "tennis"],
    "finance": ["bank", "money", "investment"],
}

# This is the word we want to classify into the closest category.
word = "startup"

# Store the average similarity score for each category.
category_scores = {}
for category, seeds in categories.items():
    # Keep only seed words that exist in the Word2Vec vocabulary.
    valid_seeds = [s for s in seeds if s in model.key_to_index and word in model.key_to_index]
    if valid_seeds:
        # Compare the target word to every valid seed word, then average the scores.
        sum_similarities = sum(model.similarity(word, s) for s in valid_seeds)
        avg_score = sum_similarities / len(valid_seeds)
        category_scores[category] = avg_score

# Print categories from most semantically similar to least similar.
print(f"Word to classify: {word}")
print("Category similarity scores:")
sorted_categories = sorted(category_scores.items(), key=lambda x: x[1], reverse=True)
for cat, score in sorted_categories:
    print(f"  {cat:12s} -> {score:.3f}")

In [ ]:
# 4) Recommendation-like behavior (find similar products/tags)
# This can be used for "users who viewed X also viewed Y" style suggestions.

# Start with one product or tag that represents the user's current interest.
seed_item = "laptop"

# Ask Word2Vec for the 10 words closest in meaning to the seed item.
recommendations = model.most_similar(seed_item, topn=10)

# Print each related item with its similarity score.
print(f"Seed item: {seed_item}")
print("Related suggestions:")
for item, score in recommendations:
    # Higher scores mean the item is more semantically related to the seed item.
    print(f"  {item:15s} -> {score:.3f}")